# 📘 Week 2 · Day 8-9: 머신러닝을 위한 수학적 기초

## 🎯 학습 목표
- Kaggle에서 꼭 필요한 **선형대수 · 확률 · 통계 · 미분/최적화**를 **직관 + 코드**로 익힌다
- 공식은 외우지 않고 **numpy로 직접 구현**하여 감을 잡는다
- 각 개념이 어디서 쓰이는지 **ML 맥락**과 연결한다

> 💡 수학을 깊게 다루기보다, **Kaggle에서 쓰는 만큼만** 빠르고 실용적으로 배웁니다.

---

## 1. 선형대수 (Linear Algebra)

### 1.1 왜 필요한가?
- 데이터는 **행렬**입니다. (row=샘플, col=피처)
- 모델 예측은 대부분 **행렬 곱** 입니다. (`y_pred = Xw + b`)
- 차원축소(PCA), 추천시스템(SVD), 딥러닝 모두 선형대수 위에 있습니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

### 1.2 벡터 · 행렬 기본

In [ ]:
# 벡터: 1차원 배열
v = np.array([1, 2, 3])
w = np.array([4, 5, 6])

# 요소별 연산
print("v + w :", v + w)
print("v * w :", v * w)

# 내적 (dot product) - 유사도 측정의 핵심
print("v · w :", np.dot(v, w), "=", (v*w).sum())

# norm (벡터 크기)
print("||v||₂:", np.linalg.norm(v))  # L2 norm
print("||v||₁:", np.linalg.norm(v, 1))  # L1 norm

In [ ]:
# 코사인 유사도 - 추천시스템·NLP에서 매일 씀
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([1, 2, 3])
b = np.array([2, 4, 6])   # a의 2배 → 완전히 같은 방향
c = np.array([-1, -2, -3]) # 반대 방향

print("cos(a, b) =", cosine_similarity(a, b))  # 1.0
print("cos(a, c) =", cosine_similarity(a, c))  # -1.0

In [ ]:
# 행렬과 행렬-벡터 곱
X = np.array([[1, 2],
              [3, 4],
              [5, 6]])   # 3x2 (샘플 3개, 피처 2개)
w = np.array([0.5, 0.3]) # 가중치 (2,)

y_pred = X @ w   # 3,
print("y_pred:", y_pred)
# 이게 바로 선형회귀의 핵심 연산!

### 1.3 행렬 연산의 의미

| 연산 | Kaggle에서의 쓰임 |
|------|--------------------|
| 전치(`.T`) | `Xᵀy` 형태로 정규방정식 풀 때 |
| 역행렬(`inv`) | 선형회귀 닫힌 해 |
| 행렬식(`det`) | 공선성 탐지 |
| 고유값/고유벡터 | PCA, 스펙트럴 클러스터링 |
| SVD | 추천, 차원축소, 노이즈 제거 |


In [ ]:
# 역행렬로 선형회귀 풀기 (Normal Equation)
# y = Xw → w = (XᵀX)⁻¹ Xᵀ y

np.random.seed(0)
X = np.random.randn(100, 3)
true_w = np.array([1.5, -2.0, 0.8])
y = X @ true_w + np.random.randn(100) * 0.1

# 정규방정식
w_hat = np.linalg.inv(X.T @ X) @ X.T @ y
print("True :", true_w)
print("Hat  :", w_hat)

In [ ]:
# SVD (Singular Value Decomposition)
# X = U Σ Vᵀ  → 데이터를 중요도 순으로 분해
X = np.random.randn(5, 3)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
print("U shape :", U.shape)
print("S (특이값):", S)   # 큰 값이 더 중요한 방향
print("Vt shape:", Vt.shape)

# 랭크 근사 (노이즈 제거·압축)
rank = 2
X_approx = U[:, :rank] @ np.diag(S[:rank]) @ Vt[:rank, :]
print("\nReconstruction error:", np.linalg.norm(X - X_approx))

---

## 2. 확률 (Probability)

### 2.1 왜 필요한가?
- 분류 문제는 `P(y | x)` 를 맞추는 것.
- 생성 모델, 베이즈 추론, 확률적 샘플링 모두 확률 기반.


In [ ]:
# 2-1. 기본 분포 3가지 (numpy 내장)

# (1) 정규분포 (가장 자주 등장)
samples_norm = np.random.normal(loc=0, scale=1, size=10000)

# (2) 이항분포 - 이진 분류 타깃
samples_binom = np.random.binomial(n=1, p=0.3, size=10000)

# (3) 포아송분포 - count 데이터 (SibSp 같은)
samples_pois = np.random.poisson(lam=3, size=10000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(samples_norm, bins=50, color='steelblue'); axes[0].set_title('Normal')
axes[1].hist(samples_binom, bins=2, color='salmon');   axes[1].set_title('Binomial p=0.3')
axes[2].hist(samples_pois, bins=15, color='green');    axes[2].set_title('Poisson λ=3')
plt.tight_layout(); plt.show()

### 2.2 기대값, 분산, 공분산

In [ ]:
x = np.random.randn(1000)
y = 2*x + np.random.randn(1000)*0.5

print("E[x]      =", x.mean())
print("Var(x)    =", x.var())
print("Cov(x,y)  =", np.cov(x, y)[0, 1])
print("Corr(x,y) =", np.corrcoef(x, y)[0, 1])

### 2.3 베이즈 정리

$$P(A|B) = \frac{P(B|A)P(A)}{P(B)}$$

**의미**: 새로운 증거(B)가 들어오면 믿음(A)을 업데이트.

**Kaggle 활용**: Naive Bayes 분류기, 베이지안 하이퍼파라미터 최적화, Target Encoding smoothing.


In [ ]:
# 암 진단 예시
# P(암) = 0.01, P(양성|암) = 0.99, P(양성|건강) = 0.05
# → 검사가 양성일 때 실제로 암일 확률은?

P_cancer = 0.01
P_pos_given_cancer = 0.99
P_pos_given_healthy = 0.05

# 전체 확률 법칙
P_pos = P_pos_given_cancer * P_cancer + P_pos_given_healthy * (1 - P_cancer)

# 베이즈
P_cancer_given_pos = (P_pos_given_cancer * P_cancer) / P_pos
print(f"양성 판정 시 실제 암 확률: {P_cancer_given_pos:.2%}")
# 놀랍게도 약 16.7% → 사전확률의 중요성

---

## 3. 통계 (Statistics)

### 3.1 ML에서 자주 쓰는 통계량
- **중심경향**: mean, median, mode
- **산포도**: variance, std, IQR
- **형태**: skewness, kurtosis
- **관계**: correlation, covariance


In [ ]:
from scipy import stats

x = np.random.exponential(1, 1000)   # 오른쪽 꼬리 긴 분포

print("mean    :", x.mean())
print("median  :", np.median(x))
print("std     :", x.std())
print("skew    :", stats.skew(x))     # 왜도 > 0 → 오른쪽 꼬리
print("kurtosis:", stats.kurtosis(x)) # 첨도

### 3.2 가설검정 (Kaggle에서 feature가 유의미한지 체크)

In [ ]:
# 두 그룹의 평균 차이 t-test (예: 생존자 vs 사망자의 Age)
np.random.seed(1)
age_survived = np.random.normal(28, 14, 342)
age_dead     = np.random.normal(31, 14, 549)

t, p = stats.ttest_ind(age_survived, age_dead)
print(f"t-statistic = {t:.3f}, p-value = {p:.4f}")
print("→ p < 0.05 이면 평균 차이가 통계적으로 유의미")

# 카이제곱 (범주형 독립성)
contingency = np.array([[233, 109],   # female survived / dead
                        [ 81, 468]])  # male   survived / dead
chi2, p, dof, exp = stats.chi2_contingency(contingency)
print(f"\nChi² = {chi2:.2f}, p = {p:.4e}")
print("→ 성별과 생존은 독립이 아님 (매우 강한 관계)")

### 3.3 중심극한정리 (CLT)

표본을 많이 뽑아 평균내면 그 평균의 분포는 **정규분포**에 가까워집니다.

**Kaggle 쓰임**: 부트스트랩, CV 표준편차 신뢰, confidence interval 추정


In [ ]:
np.random.seed(0)
# 왜도 큰 분포에서 표본 평균을 계속 뽑아보자
pop = np.random.exponential(1, 100000)   # 원본은 skewed

sample_means = [np.random.choice(pop, 50).mean() for _ in range(5000)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(pop, bins=50, color='tomato'); ax[0].set_title('Population (skewed)')
ax[1].hist(sample_means, bins=50, color='mediumseagreen'); ax[1].set_title('Sampling distribution of mean (normal!)')
plt.tight_layout(); plt.show()

---

## 4. 미분과 최적화 (Calculus & Optimization)

### 4.1 왜 필요한가?
ML은 결국 **손실함수를 최소화하는 파라미터 찾기**입니다. 이를 위해 **Gradient Descent**가 필요하고, 그 기반이 미분입니다.

### 4.2 경사하강법 직접 구현


In [ ]:
# f(w) = (w - 3)²  의 최솟값을 경사하강법으로 찾자
# f'(w) = 2(w - 3)

def f(w):   return (w - 3) ** 2
def grad(w): return 2 * (w - 3)

w = 0.0             # 초기값
lr = 0.1            # 학습률
history = [w]

for step in range(30):
    w = w - lr * grad(w)
    history.append(w)

print(f"수렴값: {w:.4f}  (정답 3.0)")

# 시각화
ws = np.linspace(-1, 7, 100)
plt.figure(figsize=(10, 4))
plt.plot(ws, f(ws), 'b-', label='f(w)')
plt.scatter(history, [f(w) for w in history], color='red', zorder=5)
plt.plot(history, [f(w) for w in history], 'r--', alpha=0.5)
plt.title('Gradient Descent')
plt.xlabel('w'); plt.ylabel('f(w)')
plt.grid(True); plt.show()

### 4.3 학습률(learning rate)의 중요성

In [ ]:
def gd(lr, steps=30):
    w = 0.0; h = [w]
    for _ in range(steps):
        w = w - lr * grad(w)
        h.append(w)
    return h

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lr in zip(axes, [0.01, 0.1, 1.01]):
    h = gd(lr)
    ax.plot(ws, f(ws), 'b-')
    ax.plot(h, [f(w) for w in h], 'ro-', alpha=0.6)
    ax.set_title(f'lr = {lr}')
    ax.set_xlim(-2, 8); ax.set_ylim(-1, 40)
plt.tight_layout(); plt.show()

# 너무 작으면 느리고, 너무 크면 발산!

### 4.4 다변수 경사하강법 (실제 ML과 동일한 형태)

In [ ]:
# 선형회귀: y = w₁x₁ + w₂x₂ + b  를 gradient descent로 학습

np.random.seed(0)
n = 200
X = np.random.randn(n, 2)
true_w = np.array([2.0, -1.5]); true_b = 0.5
y = X @ true_w + true_b + np.random.randn(n) * 0.1

# 파라미터 초기화
w = np.zeros(2); b = 0.0
lr = 0.05
losses = []

for epoch in range(300):
    y_pred = X @ w + b
    error = y_pred - y

    # 평균제곱오차의 gradient
    grad_w = (2/n) * X.T @ error
    grad_b = (2/n) * error.sum()

    w -= lr * grad_w
    b -= lr * grad_b

    loss = (error ** 2).mean()
    losses.append(loss)

print(f"True : w={true_w}, b={true_b}")
print(f"Hat  : w={w},     b={b:.3f}")

plt.figure(figsize=(8, 3))
plt.plot(losses); plt.yscale('log')
plt.title('MSE over epochs'); plt.xlabel('epoch'); plt.ylabel('loss'); plt.grid(True); plt.show()

### 4.5 Gradient Descent 변형
| 방법 | 설명 |
|------|------|
| **Batch GD** | 전체 데이터로 gradient 계산 (느리지만 안정) |
| **SGD** | 샘플 1개로 계산 (빠르지만 노이즈 많음) |
| **Mini-batch SGD** | 배치 단위로 계산 (실전 표준) |
| **Momentum** | 이전 gradient 관성 추가 (빠른 수렴) |
| **Adam** | Adaptive learning rate + Momentum (딥러닝 표준) |

XGBoost, LightGBM은 이걸 **2차 미분(Newton method)** 기반으로 발전시킨 것.

---

## 5. 손실함수 (Loss Functions)

### 5.1 회귀
- **MSE** = `mean((y - ŷ)²)` → 미분 쉬움, outlier에 민감
- **MAE** = `mean(|y - ŷ|)` → outlier에 강함
- **RMSLE** = `sqrt(mean((log(1+y) - log(1+ŷ))²))` → 큰 값 차이 완화 (Kaggle House Prices 등)

### 5.2 분류
- **Cross-Entropy** = `-y·log(p) - (1-y)·log(1-p)`


In [ ]:
# 손실함수 직접 구현
def mse(y, yhat):  return np.mean((y - yhat)**2)
def mae(y, yhat):  return np.mean(np.abs(y - yhat))
def rmsle(y, yhat):
    return np.sqrt(np.mean((np.log1p(y) - np.log1p(yhat))**2))
def log_loss(y, p, eps=1e-15):
    p = np.clip(p, eps, 1-eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

y   = np.array([100, 200, 300, 400])
yhat= np.array([110, 190, 320, 380])
print("MSE  :", mse(y, yhat))
print("MAE  :", mae(y, yhat))
print("RMSLE:", rmsle(y, yhat))

y_cls  = np.array([1, 0, 1, 1, 0])
p_hat  = np.array([0.9, 0.1, 0.8, 0.6, 0.2])
print("LogLoss:", log_loss(y_cls, p_hat))

---

## 📝 Day 8-9 체크리스트
- [ ] 벡터·행렬 연산 손으로 구현 가능
- [ ] 정규분포·이항분포 이해
- [ ] t-test, 카이제곱 가설검정 실행 경험
- [ ] 경사하강법을 numpy로 직접 구현
- [ ] MSE/MAE/LogLoss 직접 구현

이 수학 기반 위에서 다음 노트북부터 **실제 모델**을 만들기 시작합니다.